In [1]:
import sys
import torch
import torch.nn as nn
import torch.distributed as dist
import itertools
import time
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score, MulticlassConfusionMatrix
import numpy as np
from dataset_preprocessing import load_datasets,normalize_and_create_metadata,zero_pad_datasets,data_structure_gen
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
import random
import matplotlib.pyplot as plt
import boto3
import pandas as pd
import os



In [2]:
datasets = load_datasets()

In [3]:
class SDTIModel(nn.Module):
    def __init__(self,device):
        super(SDTIModel, self).__init__()
        self.device = device
    
    def forward(self,data,labels,sdti_epochs):
        datasets = int(data.shape[0]**.5)

        #sampling from a log uniform distribution to generate a lr for each dtaset
        lr_sample = torch.stack([torch.tensor(10**random.uniform(-5,-1)) for i in range(0,datasets)])
        beta_sample = torch.stack([torch.tensor((random.uniform(.85,.99),random.uniform(.98,.9999))) for i in range(0,datasets)])
        
        #Ensuring that the lr's and betas are broadcasted out into the correct shape for processing w/in the metalayer
        lrs = torch.stack([lr_sample[data] for data in range(0,datasets) for label in range(0,datasets)])
        betas = torch.stack([beta_sample[data,:] for data in range(0,datasets) for label in range(0,datasets)])
        
        output,total_memory_requirements,model_cost_data = SDTILayer(device=self.device,data=data,labels=labels, lrs=lrs, betas=betas, sdti_epochs=sdti_epochs).run()
        # output.register_hook(lambda grad: print("Gradient of MetaLayerOutput:", grad))

        output = torch.reshape(output,(datasets,datasets))

        return output,total_memory_requirements,model_cost_data

class SDTILayer(nn.Module):
    def __init__(self,device,data,labels,lrs,betas,sdti_epochs):
        super(SDTILayer,self).__init__()
        self.device = device
        self.sdti_epochs = sdti_epochs
        self.epsilon = 1e-15
        self.data = (torch.tensor(data,dtype=torch.float32)).to(self.device)
        self.labels = torch.tensor(labels,dtype=torch.int8).unsqueeze(-1).to(self.device)
        self.model_cost_data = {'records':[],'epochs':[],'sdti_epochs':[],'epoch_num':[],'sdti_epoch_num':[],'dataset':[],'labelset':[],'cost_values':[]}


        # print(self.labels.shape)
        
        #need to unsqueeze hp twice to increase dimensionality for element-wise operation when updating w&b
        self.lrs = lrs.unsqueeze(-1).unsqueeze(-1).to(self.device)
        self.beta1 = betas[:,0].unsqueeze(-1).unsqueeze(-1).to(self.device)
        self.beta2 = betas[:,1].unsqueeze(-1).unsqueeze(-1).to(self.device)

        #data is shape (289, 1980, 3072) so weights are shape (289,3072,1) & bias is (289,1)
        self.weights  = nn.init.xavier_uniform_(torch.rand(self.data.shape[0],self.data.shape[2],1)).to(self.device)
        self.bias = nn.init.xavier_uniform_(torch.rand(self.data.shape[0],1)).unsqueeze(-1).to(self.device)

        self.m_w = (torch.zeros_like(self.weights)).to(self.device)
        self.v_w = (torch.zeros_like(self.weights)).to(self.device)
        self.m_b = (torch.zeros_like(self.bias)).to(self.device)
        self.v_b = (torch.zeros_like(self.bias)).to(self.device)

        data_size = self.data.element_size() * self.data.nelement() / (1024**3)
        label_size = self.labels.element_size() * self.labels.nelement() / (1024**3)
        lr_size = self.lrs.element_size() * self.lrs.nelement() / (1024**3)
        beta_size = self.beta1.element_size() * self.beta1.nelement() / (1024**3) * 2
        weights_size = self.weights.element_size() * self.weights.nelement() / (1024**3)
        bias_size = self.bias.element_size() * self.bias.nelement() / (1024**3)
        adam_weights_size = self.m_w.element_size() * self.m_w.nelement() / (1024**3) * 2
        adam_bias_size = self.m_b.element_size() * self.m_b.nelement() / (1024**3) * 2

        self.total_memory_requirements = data_size+label_size+lr_size+beta_size+weights_size+bias_size+adam_weights_size+adam_bias_size
                
    def forward(self):
        #data is shape (289, 1980, 3072) so weights are shape (289,3072,1) & bias is (289,1)
        #a_next is shape (289, 1980, 1)
        a_next = torch.matmul(self.data,self.weights)+self.bias
        #print('data',self.data.shape,'weights',self.weights.shape,'anext',a_next.shape)
        return torch.sigmoid(a_next)

    def cost(self,output):
        #print(output.shape,self.labels.shape)
        loss =  -1 * ((self.labels * torch.log(output+self.epsilon)) + ((1 - self.labels) * torch.log(1 - output+self.epsilon)))
        cost = torch.mean(loss,dim=1)
        return cost
        
    def compute_gradients(self,output):
        
        da = -1 * ((self.labels - output) / (output * ((1 - output))+self.epsilon))
#         da.register_hook(lambda grad: print("Gradient of DA:", grad.isnan().any()))

        dsigmoid = output * (1-output+self.epsilon)
#       dsigmoid.register_hook(lambda grad: print("Gradient of DSigmoid:", grad.isnan().any()))
        
        #all I needed to do to ensure this code broadcasts properly is to carefully transpose self.data into a particular shape here
        w_grads = torch.matmul(torch.permute(self.data,(0,2,1)), da*dsigmoid).to(self.device)
#       w_grads.register_hook(lambda grad: print("Gradient of w_grads:", grad.isnan().any()))

        b_grads = torch.sum(da * dsigmoid,dim=1).to(self.device)
#         b_grads.register_hook(lambda grad: print("Gradient of b_grads:", grad.isnan().any()))
        self.total_memory_requirements += (da.element_size() * da.nelement() / (1024**3)) / self.sdti_epochs
        self.total_memory_requirements += (dsigmoid.element_size() * dsigmoid.nelement() / (1024**3)) / self.sdti_epochs

        
        return w_grads, b_grads.unsqueeze(-1)

    def adam_update(self, w_grads, b_grads, t):
#         Update first and second moments of gradients
        
        self.m_w = (self.beta1 * self.m_w + (1 - self.beta1) * w_grads)
        # self.m_w.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        self.v_w = (self.beta2 * self.v_w + (1 - self.beta2) * (w_grads ** 2))
        # self.v_w.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        self.m_b = (self.beta1 * self.m_b + (1 - self.beta1) * b_grads)
        # self.m_b.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        self.v_b = (self.beta2 * self.v_b + (1 - self.beta2) * (b_grads ** 2))
        # self.v_b.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        # Bias correction
        m_w_corrected = (self.m_w / ((1 - self.beta1 ** t)+self.epsilon))
        # m_w_corrected.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        v_w_corrected = (self.v_w / ((1 - self.beta2 ** t)+self.epsilon))
        # v_w_corrected.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        m_b_corrected = (self.m_b / ((1 - self.beta1 ** t)+self.epsilon))
        # m_b_corrected.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        v_b_corrected = (self.v_b / ((1 - self.beta2 ** t)+self.epsilon))
        # v_b_corrected.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        # Update weights and bias using Adam optimizer
        self.weights -= self.lrs * m_w_corrected / (torch.sqrt(v_w_corrected+ self.epsilon) + self.epsilon)
        # self.weights.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        self.bias -= self.lrs * m_b_corrected / (torch.sqrt(v_b_corrected+ self.epsilon) + self.epsilon)
        # self.bias.register_hook(lambda grad: grad / (torch.norm(grad) + 1e-8 if torch.norm(grad) > 0 else 1))

        self.total_memory_requirements += (m_w_corrected.element_size() * m_w_corrected.nelement() / (1024**3) * 2 ) / self.sdti_epochs
        self.total_memory_requirements += (m_b_corrected.element_size() * m_b_corrected.nelement() / (1024**3) * 2 ) / self.sdti_epochs

    def run(self):
        for i in range(1,self.sdti_epochs):
            output = self.forward()
            #stability_metric = torch.squeeze(self.lrs+(2-(self.beta1+self.beta2))).unsqueeze(-1)

            error = self.cost(output) #+stability_metric

            ##Need to capture all cost data here. Export data structure elsewhere. 

            w_grads,b_grads = self.compute_gradients(output)

            self.adam_update(w_grads,b_grads,i)

            self.model_cost_data['records'].extend([self.data.shape[1]]*error.shape[0])
            # self.model_cost_data['epochs'].extend([self.epochs]*error.shape[0])
            self.model_cost_data['sdti_epochs'].extend([self.sdti_epochs]*error.shape[0])
            # self.model_cost_data['epoch_num'].extend([epoch]*error.shape[0])
            self.model_cost_data['sdti_epoch_num'].extend([i]*error.shape[0])

            output = torch.reshape(error,(int(self.data.shape[0]**.5),int(self.data.shape[0]**.5))).detach().cpu()

            for dataset in range(0,output.shape[0]):
                for labelset in range(0,output.shape[1]):
                    self.model_cost_data['dataset'].append(dataset)
                    self.model_cost_data['labelset'].append(labelset)
                    self.model_cost_data['cost_values'].append(output[dataset,labelset].item())
                    
        self.total_memory_requirements += output.element_size() * output.nelement() / (1024**3)
        
        self.total_memory_requirements += error.element_size() * error.nelement() / (1024**3)
        self.total_memory_requirements += w_grads.element_size() * w_grads.nelement() / (1024**3)
        self.total_memory_requirements += b_grads.element_size() * b_grads.nelement() / (1024**3)
        del w_grads, b_grads

        
        return error*-1, self.total_memory_requirements,self.model_cost_data #returns cost of each dataset/labelset combination and memory requirements


In [4]:

def heatmap_gen(output):

    # Example data (replace this with your own data)

#     row_labels = [f'{i} Dataset' for i in datasets.keys()]
#     col_labels = [f'{i} Labelset' for i in datasets.keys()]
    
    # Create a heatmap using Matplotlib
    plt.imshow(nn.functional.softmax(output,dim=1).cpu().numpy(), cmap='viridis',vmin=0,vmax=0.25)  # extent sets the boundaries of the data
    plt.colorbar()  # Add a colorbar for reference
    
    # Set tick locations and labels
#     plt.xticks(range(0, len(datasets.keys())),col_labels,rotation=90)
#     plt.yticks(range(0, len(datasets.keys())), row_labels)
    plt.tick_params(left = False, right = False , labelleft = False , 
                labelbottom = False, bottom = False) 
    
    plt.xlabel("Sets of Labels")
    plt.ylabel("Datasets")
    plt.title('Class Probabilities Per Dataset-Labelset Combination')
    
    return plt.gcf()

In [5]:
def run_SDTIModel(device,train_data,train_labels,ground_truth_labels,epochs,sdti_epochs):
    start_time = time.time()

    model = SDTIModel(device)

    ground_truth_labels = torch.tensor(ground_truth_labels,dtype=torch.long).to(device)
    accuracy = MulticlassAccuracy(num_classes=int(train_data.shape[0]**.5))
    f1_score = MulticlassF1Score(num_classes=int(train_data.shape[0]**.5))

    overall_prediction = []
    for epoch in range(0,epochs):
        # Forward pass
        train_output, total_memory_requirements, model_cost_data = model(train_data,train_labels,sdti_epochs)
        model_cost_data['epochs'].extend([epochs]*len(model_cost_data['records'])) #making sure lists are the same size 
        model_cost_data['epoch_num'].extend([epoch]*len(model_cost_data['records']))

        max_vals = train_output.max(dim=1, keepdim=True).values #
        mask = (train_output == max_vals).float() #mask applies over output tensor and identifies predection per row and preserves shape of tensor
        overall_prediction.append(mask) #
        
        y_pred = torch.argmax(train_output,dim=1)
        train_acc = accuracy(y_pred.cpu(),ground_truth_labels.cpu())
        train_f1 = f1_score(y_pred.cpu(),ground_truth_labels.cpu())

    overall_prediction = torch.argmax(sum(overall_prediction),dim=1)
    overall_acc = accuracy(overall_prediction.cpu(),ground_truth_labels.cpu()).item()
    overall_f1 = f1_score(overall_prediction.cpu(),ground_truth_labels.cpu()).item()


    end_time = time.time()
    
    return overall_f1, overall_acc, total_memory_requirements, (end_time-start_time)/60, model_cost_data

In [6]:
# Data is shuffled when it is loaded into notebook
## No need to shuffle folds of data, only need to retrieve folds that are correct proportions

train_datasets, ground_truth_labels = normalize_and_create_metadata(datasets)
datasets_size = sys.getsizeof(datasets) / (1024**3)
train_datasets_size = sys.getsizeof(train_datasets) / (1024**3)

print(datasets_size)
print(train_datasets_size)
del datasets

train_datasets = zero_pad_datasets(train_datasets)

train_data,train_labels = data_structure_gen(train_datasets)

del train_datasets


train_data_size = sys.getsizeof(train_data) / (1024**3)
print(train_data_size)
train_labels_size = sys.getsizeof(train_labels) / (1024**3)
print(train_labels_size)


4.3213367462158203e-07
4.3213367462158203e-07
6.614685192704201
0.0005384236574172974


In [ ]:
num_records = [i for i in range(2000,0,-100)]
num_epochs = [i for i in range(60,0,-1) if i % 2 !=0]
num_sdti_epochs = [i for i in range(30,0,-5)]

combinations = list(itertools.product(num_records,num_epochs,num_sdti_epochs))

s3 = boto3.client('s3')
bucket_name = 'sagemaker-us-east-1-505863485541'       # Replace with your bucket name

export_test_data = {'records':[],'epochs':[],'sdti_epochs':[],'f1':[],'acc':[],'memory':[],'data':[],'ttc':[]}

export_cost_data = {'records':[],'epochs':[],'sdti_epochs':[],'epoch_num':[],'sdti_epoch_num':[],'dataset':[],'labelset':[],'cost_values':[]}

for i in combinations:
    records = i[0]
    epochs = i[1]
    sdti_epochs = i[2]
    overall_f1, overall_acc, total_memory_requirements, time_to_complete, model_cost_data = run_SDTIModel('cuda',train_data[:,:records,:],train_labels[:,:records],ground_truth_labels,epochs,sdti_epochs)


    for k in model_cost_data.keys():
        export_cost_data[k]+=model_cost_data[k]
        
    export_test_data['records'].append(records)
    export_test_data['epochs'].append(epochs)
    export_test_data['sdti_epochs'].append(sdti_epochs)
    
    export_test_data['f1'].append(overall_f1)
    export_test_data['acc'].append(overall_acc)
    export_test_data['memory'].append(total_memory_requirements) 
    export_test_data['data'].append(train_data_size+train_labels_size)
    export_test_data['ttc'].append(time_to_complete)
    
    export_test_df = pd.DataFrame(export_test_data)
    export_test_df.to_csv('TestResults.csv', index=False)

    export_cost_df = pd.DataFrame(export_cost_data)
    export_cost_df.to_csv('CostResults.csv', index=False)
    #saving export data to s3 bucket before deleting file

    s3.upload_file('TestResults.csv', bucket_name,'SDTIresults/TestResults.csv')
    s3.upload_file('CostResults.csv', bucket_name,'SDTIresults/CostResults.csv')
